# Testing, Evaluation & Explainability Notebook

This notebook performs **end-to-end testing** of a trained fetal head segmentation model.

### Requirements
- `best_model.pth` — trained model checkpoint
- Test dataset with `images/` and `masks/` directories

### Output
All results are saved under `Results/` with the following subfolders:
- `Metrics/`, `Predictions/`, `Qualitative/`, `Boundary/`, `ErrorMaps/`
- `GradCAM/`, `FeatureMaps/`, `Confidence/`, `Probability/`, `Entropy/`
- `Publication/`, `Plots/`

---
## Configuration Cell

Change only these variables to adapt to your model and dataset.

In [ ]:
MODEL_PATH = r""
DATASET_PATH = r"C:\Users\shyle\OneDrive\Documents\Projects\Fetus\HC18"
IMAGE_SIZE = 256
DEVICE = "cuda"
THRESHOLD = 0.5
RESULTS_ROOT = "Results"

---
## 1 · Install Dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless pandas numpy scikit-learn scikit-image \
    matplotlib seaborn tqdm medpy thop
!pip install -q segmentation_models_pytorch torchsummary torchinfo ptflops
import warnings
warnings.filterwarnings("ignore")

---
## 2 · Imports

In [ ]:
import os
import cv2
import time
import random
import numpy as np
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchinfo import summary
from scipy.ndimage import distance_transform_edt, binary_erosion

plt.rcParams['figure.dpi'] = 150
%matplotlib inline

---
## 3 · Dataset & Data Utilities

The dataset class, mask post-processing, augmentation, and preprocessing are identical to the training notebook.

In [ ]:
def fill_mask_border(mask):
    """Fill the white ellipse border in a binary mask to create a solid region."""
    mask = (mask * 255).astype(np.uint8) if mask.max() <= 1 else mask.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(mask)
    cv2.drawContours(filled_mask, contours, -1, color=255, thickness=cv2.FILLED)
    return (filled_mask / 255).astype(np.float32)

class FetalHeadDataset(Dataset):
    def __init__(self, images_dir, masks_dir, class_gray_values, augmentation=None, preprocessing=None):
        self.image_paths = sorted(glob(os.path.join(images_dir, '*')))
        self.mask_paths  = sorted(glob(os.path.join(masks_dir, '*'))) if masks_dir and os.path.exists(masks_dir) else []
        self.class_gray_values = class_gray_values
        self.augmentation = augmentation
        self.preprocessing = preprocessing

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, i):
        image = cv2.imread(self.image_paths[i], cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Image not found: {self.image_paths[i]}")

        if i < len(self.mask_paths):
            mask = cv2.imread(self.mask_paths[i], cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros_like(image, dtype=np.float32)
            else:
                mask = fill_mask_border(mask)
                if self.class_gray_values:
                    mask = np.isin(mask * 255, self.class_gray_values).astype('float32')
                else:
                    mask = mask.astype('float32')
        else:
            mask = np.zeros_like(image, dtype=np.float32)

        image = image.astype('float32') / 255.0
        image = np.expand_dims(image, -1)
        mask  = np.expand_dims(mask, -1)

        if self.augmentation:
            augmented = self.augmentation(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        if self.preprocessing:
            processed = self.preprocessing(image=image, mask=mask)
            image, mask = processed['image'], processed['mask']

        if isinstance(mask, torch.Tensor):
            if mask.ndim == 4 and mask.shape[-1] == 1:
                mask = mask.squeeze(-1)
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
            if mask.ndim == 3 and mask.shape[0] != 1 and mask.shape[-1] == 1:
                mask = mask.permute(2, 0, 1)
        elif isinstance(mask, np.ndarray):
            if mask.ndim == 4 and mask.shape[-1] == 1:
                mask = np.squeeze(mask, axis=-1)
            if mask.ndim == 2:
                mask = np.expand_dims(mask, 0)
            if mask.ndim == 3 and mask.shape[0] != 1 and mask.shape[-1] == 1:
                mask = np.transpose(mask, (2, 0, 1))
            mask = torch.from_numpy(mask).float()

        assert mask.shape == (1, 256, 256), f"Mask shape: {mask.shape}"

        if isinstance(image, torch.Tensor):
            if image.ndim == 4 and image.shape[-1] == 1:
                image = image.squeeze(-1)
            if image.ndim == 2:
                image = image.unsqueeze(0)
            if image.ndim == 3 and image.shape[0] != 1 and image.shape[-1] == 1:
                image = image.permute(2, 0, 1)
        elif isinstance(image, np.ndarray):
            if image.ndim == 4 and image.shape[-1] == 1:
                image = np.squeeze(image, axis=-1)
            if image.ndim == 2:
                image = np.expand_dims(image, 0)
            if image.ndim == 3 and image.shape[0] != 1 and image.shape[-1] == 1:
                image = np.transpose(image, (2, 0, 1))
            image = torch.from_numpy(image).float()

        assert image.shape == (1, 256, 256), f"Image shape: {image.shape}"

        return image, mask

---
## 4 · Validation Augmentation & Preprocessing

Exactly as used during validation / testing in the training notebook.

In [ ]:
def get_validation_augmentation():
    return A.Compose([
        A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE),
        A.CenterCrop(IMAGE_SIZE, IMAGE_SIZE)
    ])

def get_preprocessing():
    return A.Compose([
        A.Normalize(mean=(0.0,), std=(1.0,), max_pixel_value=1.0),
        ToTensorV2()
    ])

---
## 5 · Auto-Detect Dataset Structure & Create DataLoader

Supports:
- `dataset/images/` + `dataset/masks/` (flat structure)
- HC18 structure: `dataset/test/images/` + `dataset/test/masks/`
- `dataset/test/` containing `images/` and `masks/`

In [ ]:
def auto_detect_dataset(base_path):
    """Auto-detect dataset structure and return (images_dir, masks_dir)."""
    candidates = [
        (os.path.join(base_path, 'images'), os.path.join(base_path, 'masks')),
        (os.path.join(base_path, 'test', 'images'), os.path.join(base_path, 'test', 'masks')),
        (os.path.join(base_path, 'val', 'images'), os.path.join(base_path, 'val', 'masks')),
    ]
    for img_dir, msk_dir in candidates:
        if os.path.exists(img_dir) and os.path.exists(msk_dir) and any(glob(os.path.join(img_dir, '*'))):
            return img_dir, msk_dir
    raise FileNotFoundError(
        f"Could not find test images under {base_path}. "
        f"Expected 'images/' and 'masks/' in one of these structures:\n"
        f"  {base_path}/images/  {base_path}/masks/\n"
        f"  {base_path}/test/images/  {base_path}/test/masks/\n"
        f"  {base_path}/val/images/  {base_path}/val/masks/"
    )

select_class_gray_values = [255]
test_images_dir, test_masks_dir = auto_detect_dataset(DATASET_PATH)
print(f"Images: {test_images_dir}")
print(f"Masks:  {test_masks_dir}")

test_dataset = FetalHeadDataset(
    test_images_dir, test_masks_dir,
    class_gray_values=select_class_gray_values,
    augmentation=get_validation_augmentation(),
    preprocessing=get_preprocessing()
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
print(f"Test samples: {len(test_dataset)}")

---
## 6 · Model Definition

**Do NOT edit this cell unless you are adapting to a different architecture.**

Exactly the same model definition as the training notebook.
Select the desired model by uncommenting one block.

In [ ]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')
sys.path.insert(0, os.path.join(os.getcwd(), 'Experiments'))
sys.path.insert(0, r'C:\Users\shyle\OneDrive\Documents\Projects\Fetus\Experiments')
from models_library import (
    nnUNet,
    ResAttnUNet,
    nnUNet_ResAttn,
    nnUNet_MSFFBridge_ResAttn,
    build_UNet_MiTB2,
    UNet_Hybrid,
    msunet
)

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# =========================================================================
#  SELECT YOUR MODEL ARCHITECTURE (Uncomment ONE block below)
# =========================================================================

# --- 1) nnUNet ---
# model = nnUNet(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "nnUNet"

# --- 2) ResAttnUNet ---
# model = ResAttnUNet(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "ResAttnUNet"

# --- 3) nnUNet_ResAttn ---
# model = nnUNet_ResAttn(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "nnUNet_ResAttn"

# --- 4) nnUNet_MSFFBridge_ResAttn ---
# model = nnUNet_MSFFBridge_ResAttn(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "nnUNet_MSFFBridge_ResAttn"

# --- 5) UNet_MiTB2 ---
# model = build_UNet_MiTB2(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "UNet_MiTB2"

# --- 6) UNet_Hybrid ---
# model = UNet_Hybrid(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "UNet_Hybrid"

# --- 7) msunet ---
# model = msunet(in_ch=1, out_ch=1).to(device)
# MODEL_NAME = "msunet"

assert model is not None, "You must define a model!"
print(f"Model architecture: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
try:
    summary(model, input_size=(1, 1, IMAGE_SIZE, IMAGE_SIZE))
except Exception:
    pass

---
## 7 · Load Trained Weights

Loads the best model checkpoint, extracts the state dict, and sets the model to evaluation mode.

In [ ]:
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')} "
          f"(Val Dice: {checkpoint.get('best_val_dice', 'N/A'):.4f})")
else:
    model.load_state_dict(checkpoint)
    print("Loaded model weights.")
model.eval()
print("Model set to evaluation mode.")

---
## 8 · Evaluation Metrics Definition

All pixel-level (Accuracy, Dice, IoU, Precision, Recall, F1, Specificity, Sensitivity) and surface-distance metrics used for evaluation.

In [ ]:
def to_float(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().item()
    return float(x)

def safe_div(numerator, denominator):
    return (numerator + 1e-7) / (denominator + 1e-7) if denominator != 0 else 0.0

def dice_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    intersection = (outputs * labels).sum()
    total = outputs.sum() + labels.sum()
    return safe_div(2 * intersection, total)

def iou_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    intersection = (outputs * labels).sum()
    union = (outputs + labels).clamp(0, 1).sum()
    return safe_div(intersection, union)

def precision_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    tp = (outputs * labels).sum()
    fp = (outputs * (1 - labels)).sum()
    return safe_div(tp, tp + fp)

def recall_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    tp = (outputs * labels).sum()
    fn = ((1 - outputs) * labels).sum()
    return safe_div(tp, tp + fn)

def accuracy_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    correct = (outputs == labels).float().sum()
    total = torch.numel(outputs)
    return safe_div(correct, total)

def f1_score_pytorch(outputs, labels, threshold=0.5):
    prec = precision_pytorch(outputs, labels, threshold)
    rec = recall_pytorch(outputs, labels, threshold)
    return safe_div(2 * prec * rec, prec + rec)

def specificity_pytorch(outputs, labels, threshold=0.5):
    outputs = (outputs > threshold).float()
    labels = (labels > 0.5).float()
    tn = ((1 - outputs) * (1 - labels)).sum()
    fp = (outputs * (1 - labels)).sum()
    return safe_div(tn, tn + fp)

def sensitivity_pytorch(outputs, labels, threshold=0.5):
    return recall_pytorch(outputs, labels, threshold)

def compute_surface_distances(pred_mask, gt_mask):
    """Compute HD95 and ASSD surface distances between binary masks."""
    pred_bin = pred_mask.astype(bool)
    gt_bin = gt_mask.astype(bool)

    if not pred_bin.any() and not gt_bin.any():
        return 0.0, 0.0
    if not pred_bin.any() or not gt_bin.any():
        return float('nan'), float('nan')

    dt_pred = distance_transform_edt(~pred_bin)
    dt_gt = distance_transform_edt(~gt_bin)

    pred_surface = pred_bin ^ binary_erosion(pred_bin, iterations=1)
    gt_surface = gt_bin ^ binary_erosion(gt_bin, iterations=1)

    if not pred_surface.any():
        pred_surface = pred_bin
    if not gt_surface.any():
        gt_surface = gt_bin

    dist_pred_to_gt = dt_gt[pred_surface]
    dist_gt_to_pred = dt_pred[gt_surface]

    all_distances = np.concatenate([dist_pred_to_gt, dist_gt_to_pred])
    hd95 = float(np.percentile(all_distances, 95))
    assd = float((dist_pred_to_gt.mean() + dist_gt_to_pred.mean()) / 2.0)

    return hd95, assd

---
## 9 · Full Test Set Inference

Run inference on the entire test dataset, compute metrics per sample, and measure inference time.

In [ ]:
per_sample_results = []
inference_times = []

model.eval()
with torch.no_grad():
    for idx in tqdm(range(len(test_dataset)), desc="Testing"):
        img_tensor, mask_tensor = test_dataset[idx]
        img_tensor = img_tensor.unsqueeze(0).to(device)
        mask_tensor = mask_tensor.unsqueeze(0).to(device)
        img_path = test_dataset.image_paths[idx]
        img_name = os.path.basename(img_path)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        t_start = time.perf_counter()
        logits = model(img_tensor)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t_end = time.perf_counter()
        infer_time = (t_end - t_start) * 1000
        inference_times.append(infer_time)

        probs = torch.sigmoid(logits)
        pred_mask = (probs > THRESHOLD).float()

        d = to_float(dice_pytorch(probs, mask_tensor, THRESHOLD))
        i = to_float(iou_pytorch(probs, mask_tensor, THRESHOLD))
        p = to_float(precision_pytorch(probs, mask_tensor, THRESHOLD))
        r = to_float(recall_pytorch(probs, mask_tensor, THRESHOLD))
        a = to_float(accuracy_pytorch(probs, mask_tensor, THRESHOLD))
        f = to_float(f1_score_pytorch(probs, mask_tensor, THRESHOLD))
        spec = to_float(specificity_pytorch(probs, mask_tensor, THRESHOLD))
        sens = to_float(sensitivity_pytorch(probs, mask_tensor, THRESHOLD))

        pred_np = pred_mask[0, 0].cpu().numpy()
        mask_np = (mask_tensor > 0.5)[0, 0].cpu().numpy()
        hd95, assd = compute_surface_distances(pred_np, mask_np)

        per_sample_results.append({
            'idx': idx,
            'img_path': img_path,
            'img_name': img_name,
            'dice': d,
            'iou': i,
            'precision': p,
            'recall': r,
            'accuracy': a,
            'f1': f,
            'specificity': spec,
            'sensitivity': sens,
            'hd95': hd95,
            'assd': assd,
            'image_tensor': img_tensor.cpu(),
            'mask_tensor': mask_tensor.cpu(),
            'pred_mask': pred_np,
            'prob_map': probs[0, 0].cpu().numpy(),
        })

print(f"Inference complete. Processed {len(per_sample_results)} samples.")

---
## 10 · Create Results Directory Structure

In [ ]:
subdirs = [
    'Metrics', 'Predictions', 'Qualitative', 'Boundary', 'ErrorMaps',
    'GradCAM', 'FeatureMaps', 'Confidence', 'Probability', 'Entropy',
    'Publication', 'Plots'
]
for sub in subdirs:
    os.makedirs(os.path.join(RESULTS_ROOT, sub), exist_ok=True)
print(f"Results directory structure created under: {os.path.abspath(RESULTS_ROOT)}")

---
## 11 · Save Predicted Masks

Saves binary prediction masks as PNG files in `Results/Predictions/`.

In [ ]:
for sample in tqdm(per_sample_results, desc="Saving Predictions"):
    mask_u8 = (sample['pred_mask'] * 255).astype(np.uint8)
    save_path = os.path.join(RESULTS_ROOT, 'Predictions', sample['img_name'])
    cv2.imwrite(save_path, mask_u8)
print(f"Predictions saved to {os.path.join(RESULTS_ROOT, 'Predictions')}/")

---
## 12 · Save Metrics CSV

Per-image metrics saved to `Results/Metrics.csv` with summary statistics.

In [ ]:
df_metrics = pd.DataFrame([{
    'Image': s['img_name'],
    'Dice': s['dice'],
    'IoU': s['iou'],
    'Precision': s['precision'],
    'Recall': s['recall'],
    'Accuracy': s['accuracy'],
    'F1': s['f1'],
    'Specificity': s['specificity'],
    'Sensitivity': s['sensitivity'],
    'HD95': s['hd95'],
    'ASSD': s['assd'],
} for s in per_sample_results])

summary_row = {
    'Image': '---SUMMARY---',
    'Dice': f"{df_metrics['Dice'].mean():.4f} +/- {df_metrics['Dice'].std():.4f}",
    'IoU': f"{df_metrics['IoU'].mean():.4f} +/- {df_metrics['IoU'].std():.4f}",
    'Precision': f"{df_metrics['Precision'].mean():.4f} +/- {df_metrics['Precision'].std():.4f}",
    'Recall': f"{df_metrics['Recall'].mean():.4f} +/- {df_metrics['Recall'].std():.4f}",
    'Accuracy': f"{df_metrics['Accuracy'].mean():.4f} +/- {df_metrics['Accuracy'].std():.4f}",
    'F1': f"{df_metrics['F1'].mean():.4f} +/- {df_metrics['F1'].std():.4f}",
    'Specificity': f"{df_metrics['Specificity'].mean():.4f} +/- {df_metrics['Specificity'].std():.4f}",
    'Sensitivity': f"{df_metrics['Sensitivity'].mean():.4f} +/- {df_metrics['Sensitivity'].std():.4f}",
    'HD95': f"{df_metrics['HD95'].mean():.4f} +/- {df_metrics['HD95'].std():.4f}",
    'ASSD': f"{df_metrics['ASSD'].mean():.4f} +/- {df_metrics['ASSD'].std():.4f}",
}
df_with_summary = pd.concat([
    df_metrics,
    pd.DataFrame([summary_row])
], ignore_index=True)

csv_path = os.path.join(RESULTS_ROOT, 'Metrics', 'Metrics.csv')
df_with_summary.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

---
## 13 · Qualitative Analysis (5 Representative Images)

Selects 5 representative images: best, worst, and 3 from the middle range.
For each: display Original, Ground Truth, Predicted Mask, Overlay, and Prediction Probability Map.

In [ ]:
def select_representative_samples(results, n=5):
    """Select n representative samples spanning the performance range."""
    sorted_samples = sorted(results, key=lambda x: x['dice'])
    total = len(sorted_samples)
    if total <= n:
        return sorted_samples
    indices = [0, total // 4, total // 2, 3 * total // 4, total - 1]
    return [sorted_samples[i] for i in indices[:n]]

repr_samples = select_representative_samples(per_sample_results, n=5)

def draw_overlay(image, pred_mask, color=(0, 255, 0)):
    """Draw prediction contour overlay on image."""
    img_rgb = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    contours, _ = cv2.findContours(
        (pred_mask * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if contours:
        cv2.drawContours(img_rgb, contours, -1, color, 2)
    return img_rgb

for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    gt_np = sample['mask_tensor'][0, 0].numpy()
    pred_np = sample['pred_mask']
    prob_np = sample['prob_map']

    fig, axes = plt.subplots(1, 5, figsize=(20, 4), dpi=300)
    fig.suptitle(f"Qualitative Analysis — {sample['img_name']} (Dice: {sample['dice']:.4f}, IoU: {sample['iou']:.4f})",
                 fontsize=12, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(gt_np, cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(pred_np, cmap='gray')
    axes[2].set_title("Predicted Mask")
    axes[2].axis('off')

    overlay = draw_overlay(img_np, pred_np)
    axes[3].imshow(overlay)
    axes[3].set_title("Overlay")
    axes[3].axis('off')

    im = axes[4].imshow(prob_np, cmap='viridis', vmin=0, vmax=1)
    axes[4].set_title("Probability Map")
    axes[4].axis('off')
    plt.colorbar(im, ax=axes[4], fraction=0.046, pad=0.04)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'Qualitative', f"qual_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 14 · Boundary Comparison

Compare ground truth boundary vs predicted boundary with combined overlay.

In [ ]:
def extract_boundary(binary_mask):
    """Extract boundary pixels from a binary mask."""
    if binary_mask.sum() == 0:
        return np.zeros_like(binary_mask, dtype=np.uint8)
    eroded = binary_erosion(binary_mask.astype(bool), iterations=1)
    boundary = binary_mask.astype(bool) ^ eroded
    return boundary.astype(np.uint8)

for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    gt_np = (sample['mask_tensor'][0, 0].numpy() > 0.5).astype(np.uint8)
    pred_np = sample['pred_mask']

    gt_boundary = extract_boundary(gt_np)
    pred_boundary = extract_boundary(pred_np)

    img_rgb = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    combined_overlay = img_rgb.copy()
    combined_overlay[gt_boundary > 0] = [0, 255, 0]
    combined_overlay[pred_boundary > 0] = [0, 0, 255]
    both = (gt_boundary > 0) & (pred_boundary > 0)
    combined_overlay[both] = [255, 255, 0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
    fig.suptitle(f"Boundary Comparison — {sample['img_name']}", fontsize=13, fontweight='bold')

    axes[0].imshow(gt_boundary, cmap='gray')
    axes[0].set_title("Ground Truth Boundary")
    axes[0].axis('off')

    axes[1].imshow(pred_boundary, cmap='gray')
    axes[1].set_title("Prediction Boundary")
    axes[1].axis('off')

    axes[2].imshow(combined_overlay)
    axes[2].set_title("Combined (Green: GT, Red: Pred, Yellow: Both)")
    axes[2].axis('off')

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'Boundary', f"boundary_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 15 · Error Maps

Color-coded error maps: True Positive (green), False Positive (red), False Negative (blue).

In [ ]:
for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    gt_np = (sample['mask_tensor'][0, 0].numpy() > 0.5).astype(np.uint8)
    pred_np = sample['pred_mask']

    tp = (pred_np == 1) & (gt_np == 1)
    fp = (pred_np == 1) & (gt_np == 0)
    fn = (pred_np == 0) & (gt_np == 1)

    error_map = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
    error_map[tp] = [0, 255, 0]
    error_map[fp] = [255, 0, 0]
    error_map[fn] = [0, 0, 255]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
    fig.suptitle(f"Error Map — {sample['img_name']} (Dice: {sample['dice']:.4f})",
                 fontsize=13, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(error_map)
    axes[1].set_title("Error Map (TP: Green, FP: Red, FN: Blue)")
    axes[1].axis('off')

    legend_patches = [
        mpatches.Patch(color='green', label='True Positive'),
        mpatches.Patch(color='red', label='False Positive'),
        mpatches.Patch(color='blue', label='False Negative'),
    ]
    axes[1].legend(handles=legend_patches, loc='lower right', fontsize=8)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'ErrorMaps', f"errmap_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 16 · Grad-CAM

Computes Grad-CAM heatmaps using the last meaningful convolutional layer of the model.

In [ ]:
def find_gradcam_target_layer(model):
    """Automatically locate the last meaningful feature extraction layer for Grad-CAM."""
    if hasattr(model, 'decoder') and hasattr(model.decoder, 'blocks'):
        try:
            last_block = model.decoder.blocks[-1]
            convs = [m for m in last_block.modules() if isinstance(m, nn.Conv2d)]
            if convs:
                print(f"Grad-CAM Layer: Decoder last block Conv2d")
                return convs[-1]
        except Exception:
            pass

    conv_modules = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_modules.append((name, module))

    if conv_modules:
        swin_pre_convs = [m for name, m in conv_modules if 'bridge' in name.lower() or 'pre_swin' in name.lower()]
        if swin_pre_convs:
            print(f"Grad-CAM Layer: Pre-Swin Transformer Conv layer")
            return swin_pre_convs[-1]
        last_name, last_conv = conv_modules[-1]
        print(f"Grad-CAM Layer: '{last_name}' (last Conv2d layer)")
        return last_conv

    raise ValueError("No Conv2d layer found in model for Grad-CAM!")

def compute_gradcam(model, input_tensor, target_layer):
    """Compute normalized Grad-CAM heatmap (2D numpy array [0, 1])."""
    activations = []
    gradients = []

    def forward_hook(module, input, output):
        activations.append(output.detach())

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].detach())

    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_full_backward_hook(backward_hook)

    model.zero_grad()
    x = input_tensor.clone().detach().requires_grad_(True)
    logits = model(x)
    probs = torch.sigmoid(logits)

    target_score = probs.sum()
    target_score.backward()

    fh.remove()
    bh.remove()

    if not activations or not gradients:
        return np.zeros((input_tensor.shape[2], input_tensor.shape[3]))

    act = activations[0]
    grad = gradients[0]

    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * act).sum(dim=1))[0].cpu().numpy()

    if cam.max() > cam.min():
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    else:
        cam = np.zeros_like(cam)

    cam_resized = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))
    return cam_resized

gradcam_layer = find_gradcam_target_layer(model)
print(f"Grad-CAM target layer: {gradcam_layer}")

In [ ]:
for i, sample in enumerate(repr_samples):
    img_tensor = sample['image_tensor'].to(device)
    img_np = img_tensor[0, 0].cpu().numpy()

    cam = compute_gradcam(model, img_tensor, gradcam_layer)

    img_rgb = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    cam_u8 = (cam * 255).astype(np.uint8)
    cam_jet = cv2.applyColorMap(cam_u8, cv2.COLORMAP_JET)
    cam_jet_rgb = cv2.cvtColor(cam_jet, cv2.COLOR_BGR2RGB)
    overlay = np.uint8(np.clip(0.5 * (img_rgb / 255.0) + 0.5 * (cam_jet_rgb / 255.0), 0, 1) * 255)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
    fig.suptitle(f"Grad-CAM — {sample['img_name']} (Dice: {sample['dice']:.4f})",
                 fontsize=13, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(cam_jet_rgb)
    axes[1].set_title("Grad-CAM Heatmap")
    axes[1].axis('off')

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis('off')

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'GradCAM', f"gradcam_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 17 · Feature Maps

Visualize intermediate encoder feature activations.

In [ ]:
def visualize_feature_maps(model, sample_tensor, save_dir, prefix=""):
    """Extract and visualize feature maps from encoder and other Conv2d modules."""
    activations = {}
    hooks = []

    for name, module in model.named_modules():
        if 'encoder' in name.lower() and isinstance(module, nn.Conv2d):
            if 'Encoder' not in activations:
                activations['Encoder'] = (name, module)
        elif 'decoder' in name.lower() and isinstance(module, nn.Conv2d):
            if 'Decoder' not in activations:
                activations['Decoder'] = (name, module)

    act_maps = {}
    for comp_name, (mod_name, module) in activations.items():
        def hook_fn(m, i, o, key=comp_name):
            act_maps[key] = o.detach()
        hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(sample_tensor.to(device))

    for h in hooks:
        h.remove()

    for comp_name, feat_map in act_maps.items():
        feat_np = feat_map[0].cpu().numpy()
        n_channels = min(16, feat_np.shape[0])
        n_cols = 4
        n_rows = int(np.ceil(n_channels / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows), dpi=300)
        fig.suptitle(f"Feature Maps — {comp_name}", fontsize=14, fontweight='bold')
        axes = axes.flatten()

        for idx in range(n_cols * n_rows):
            if idx < n_channels:
                ch_map = feat_np[idx]
                ch_norm = (ch_map - ch_map.min()) / (ch_map.max() - ch_map.min() + 1e-8)
                axes[idx].imshow(ch_norm, cmap='viridis')
                axes[idx].set_title(f"Ch {idx+1}", fontsize=8)
            axes[idx].axis('off')

        plt.tight_layout()
        save_p = os.path.join(save_dir, f"{prefix}feature_map_{comp_name.lower()}.png")
        fig.savefig(save_p, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved Feature Maps: {save_p}")

sample_tensor = repr_samples[0]['image_tensor'].to(device)
visualize_feature_maps(model, sample_tensor, os.path.join(RESULTS_ROOT, 'FeatureMaps'), prefix="")

---
## 18 · Confidence Maps

Pixel-wise confidence = prediction probability (maps to [0, 1] scale).
A value close to 1 indicates high confidence in the prediction.

In [ ]:
for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    prob_np = sample['prob_map']

    confidence = np.abs(prob_np - 0.5) * 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
    fig.suptitle(f"Confidence Map — {sample['img_name']}", fontsize=13, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(prob_np, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title("Prediction Probability")
    axes[1].axis('off')
    plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04)

    im = axes[2].imshow(confidence, cmap='RdYlGn', vmin=0, vmax=1)
    axes[2].set_title("Confidence (0=Uncertain, 1=Certain)")
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'Confidence', f"confidence_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 19 · Probability Maps

Display raw prediction probabilities before thresholding.

In [ ]:
for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    prob_np = sample['prob_map']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
    fig.suptitle(f"Probability Map — {sample['img_name']} (Threshold: {THRESHOLD})",
                 fontsize=13, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    im = axes[1].imshow(prob_np, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title("Prediction Probability")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'Probability', f"prob_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 20 · Entropy Maps (Uncertainty)

Pixel-wise entropy = -p*log(p) - (1-p)*log(1-p).
High entropy indicates high uncertainty in the prediction.

In [ ]:
def compute_entropy(prob_map, eps=1e-8):
    """Compute pixel-wise binary entropy."""
    p = np.clip(prob_map, eps, 1 - eps)
    return -p * np.log(p) - (1 - p) * np.log(1 - p)

for i, sample in enumerate(repr_samples):
    img_np = sample['image_tensor'][0, 0].numpy()
    prob_np = sample['prob_map']
    entropy = compute_entropy(prob_np)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
    fig.suptitle(f"Entropy (Uncertainty) Map — {sample['img_name']}", fontsize=13, fontweight='bold')

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(prob_np, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title("Prediction Probability")
    axes[1].axis('off')
    plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04)

    im = axes[2].imshow(entropy, cmap='hot')
    axes[2].set_title("Entropy (Uncertainty)")
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_ROOT, 'Entropy', f"entropy_{i+1}_{sample['img_name']}")
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

---
## 21 · Performance Analysis

Measure inference time, FPS, FLOPs (GFLOPs / GMACs), model size, and parameter count.

In [ ]:
inference_times_ms = np.array(inference_times)
avg_inference_ms = float(np.mean(inference_times_ms))
std_inference_ms = float(np.std(inference_times_ms))
fps = 1000.0 / avg_inference_ms if avg_inference_ms > 0 else 0

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model_size_mb = 0
if os.path.exists(MODEL_PATH):
    model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)

# Calculate FLOPs & MACs
gflops, gmacs = float('nan'), float('nan')
try:
    from thop import profile
    dummy_input = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device)
    macs, _ = profile(model, inputs=(dummy_input,), verbose=False)
    gmacs = macs / 1e9
    gflops = (macs * 2) / 1e9
except Exception as e1:
    try:
        from ptflops import get_model_complexity_info
        macs, _ = get_model_complexity_info(model, (1, IMAGE_SIZE, IMAGE_SIZE), as_strings=False, print_per_layer_stat=False, verbose=False)
        gmacs = macs / 1e9
        gflops = (macs * 2) / 1e9
    except Exception as e2:
        print(f"Could not compute FLOPs: {e1} | {e2}")

print("=" * 55)
print("  PERFORMANCE ANALYSIS")
print("=" * 55)
print(f"  {'Total Parameters':<30} {total_params:>15,}")
print(f"  {'Trainable Parameters':<30} {trainable_params:>15,}")
print(f"  {'FLOPs (GFLOPs)':<30} {gflops:>15.4f}")
print(f"  {'MACs (GMACs)':<30} {gmacs:>15.4f}")
print(f"  {'Model Size (MB)':<30} {model_size_mb:>15.2f}")
print(f"  {'Avg Inference Time (ms)':<30} {avg_inference_ms:>15.2f}")
print(f"  {'Std Inference Time (ms)':<30} {std_inference_ms:>15.2f}")
print(f"  {'FPS':<30} {fps:>15.2f}")
print("=" * 55)


---
## 22 · Distribution Plots

Dice and IoU distributions, boxplots, and histograms of all metrics.

In [ ]:
metric_keys = ['Dice', 'IoU', 'Precision', 'Recall', 'Accuracy', 'F1', 'Specificity', 'Sensitivity']
metric_values = {k: [s[k.lower()] for s in per_sample_results] for k in metric_keys}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300)
fig.suptitle(f"Metric Distributions — {MODEL_NAME}", fontsize=16, fontweight='bold')

axes[0, 0].hist(metric_values['Dice'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(np.mean(metric_values['Dice']), color='red', linestyle='--', label=f"Mean: {np.mean(metric_values['Dice']):.4f}")
axes[0, 0].set_title("Dice Distribution")
axes[0, 0].set_xlabel("Dice")
axes[0, 0].legend()

axes[0, 1].hist(metric_values['IoU'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(np.mean(metric_values['IoU']), color='red', linestyle='--', label=f"Mean: {np.mean(metric_values['IoU']):.4f}")
axes[0, 1].set_title("IoU Distribution")
axes[0, 1].set_xlabel("IoU")
axes[0, 1].legend()

box_data = [metric_values[k] for k in ['Dice', 'IoU', 'F1', 'Precision', 'Recall', 'Accuracy']]
bp = axes[0, 2].boxplot(box_data, patch_artist=True)
axes[0, 2].set_xticklabels(['Dice', 'IoU', 'F1', 'Precision', 'Recall', 'Accuracy'])
for patch, color in zip(bp['boxes'], ['steelblue', 'coral', 'lightgreen', 'gold', 'lightcoral', 'lightblue']):
    patch.set_facecolor(color)
axes[0, 2].set_title("Boxplots of Metrics")
axes[0, 2].tick_params(axis='x', rotation=45)

all_vals = [v for kv in metric_values.values() for v in kv]
all_labels = []
for k in metric_keys:
    all_labels.extend([k] * len(metric_values[k]))

axes[1, 0].hist(metric_values['Precision'], bins=30, color='gold', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(np.mean(metric_values['Precision']), color='red', linestyle='--', label=f"Mean: {np.mean(metric_values['Precision']):.4f}")
axes[1, 0].set_title("Precision Distribution")
axes[1, 0].set_xlabel("Precision")
axes[1, 0].legend()

axes[1, 1].hist(metric_values['Recall'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(np.mean(metric_values['Recall']), color='red', linestyle='--', label=f"Mean: {np.mean(metric_values['Recall']):.4f}")
axes[1, 1].set_title("Recall Distribution")
axes[1, 1].set_xlabel("Recall")
axes[1, 1].legend()

axes[1, 2].hist(metric_values['F1'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 2].axvline(np.mean(metric_values['F1']), color='red', linestyle='--', label=f"Mean: {np.mean(metric_values['F1']):.4f}")
axes[1, 2].set_title("F1 Distribution")
axes[1, 2].set_xlabel("F1")
axes[1, 2].legend()

plt.tight_layout()
save_path = os.path.join(RESULTS_ROOT, 'Plots', 'distribution_plots.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")
plt.close(fig)

---
## 23 · Best & Worst Predictions

Automatically identifies top-5 (highest Dice) and bottom-5 (lowest Dice) images.

In [ ]:
sorted_by_dice = sorted(per_sample_results, key=lambda x: x['dice'], reverse=True)
top5 = sorted_by_dice[:5]
bottom5 = sorted_by_dice[-5:]

print("Top 5 (Highest Dice):")
for s in top5:
    print(f"  {s['img_name']}: Dice={s['dice']:.4f}, IoU={s['iou']:.4f}")
print("\nBottom 5 (Lowest Dice):")
for s in bottom5:
    print(f"  {s['img_name']}: Dice={s['dice']:.4f}, IoU={s['iou']:.4f}")

def plot_prediction_comparison(samples, title, filename):
    n = len(samples)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n), dpi=300)
    fig.suptitle(title, fontsize=16, fontweight='bold')

    for idx, sample in enumerate(samples):
        img_np = sample['image_tensor'][0, 0].numpy()
        gt_np = sample['mask_tensor'][0, 0].numpy()
        pred_np = sample['pred_mask']
        overlay = draw_overlay(img_np, pred_np)

        row_axes = axes[idx] if n > 1 else axes
        row_axes[0].imshow(img_np, cmap='gray')
        row_axes[0].set_title(f"{sample['img_name']}", fontsize=10)
        row_axes[0].axis('off')

        row_axes[1].imshow(gt_np, cmap='gray')
        row_axes[1].set_title("Ground Truth")
        row_axes[1].axis('off')

        row_axes[2].imshow(pred_np, cmap='gray')
        row_axes[2].set_title("Prediction")
        row_axes[2].axis('off')

        row_axes[3].imshow(overlay)
        row_axes[3].set_title(f"Overlay (Dice: {sample['dice']:.4f})")
        row_axes[3].axis('off')

    plt.tight_layout()
    save_p = os.path.join(RESULTS_ROOT, 'Plots', filename)
    fig.savefig(save_p, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_p}")
    plt.close(fig)

plot_prediction_comparison(top5, "Top 5 Best Predictions (Highest Dice)", "best_5_predictions.png")
plot_prediction_comparison(bottom5, "Bottom 5 Worst Predictions (Lowest Dice)", "worst_5_predictions.png")

---
## 24 · Publication-Ready Figure

Single figure containing: Original, Ground Truth, Prediction, Boundary Overlay, Error Map, Grad-CAM, Confidence Map, Entropy Map.

In [ ]:
best_sample = sorted_by_dice[0]
img_tensor = best_sample['image_tensor'].to(device)
img_np = img_tensor[0, 0].cpu().numpy()
gt_np = (best_sample['mask_tensor'][0, 0].numpy() > 0.5).astype(np.uint8)
pred_np = best_sample['pred_mask']
prob_np = best_sample['prob_map']

cam = compute_gradcam(model, img_tensor, gradcam_layer)
entropy = compute_entropy(prob_np)
confidence = np.abs(prob_np - 0.5) * 2

img_rgb = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
cam_u8 = (cam * 255).astype(np.uint8)
cam_jet = cv2.cvtColor(cv2.applyColorMap(cam_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
cam_overlay = np.uint8(np.clip(0.5 * (img_rgb / 255.0) + 0.5 * (cam_jet / 255.0), 0, 1) * 255)

overlay = draw_overlay(img_np, pred_np, color=(0, 255, 0))
gt_boundary = extract_boundary(gt_np)
pred_boundary = extract_boundary(pred_np)
boundary_rgb = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
boundary_rgb[gt_boundary > 0] = [0, 255, 0]
boundary_rgb[pred_boundary > 0] = [0, 0, 255]
both = (gt_boundary > 0) & (pred_boundary > 0)
boundary_rgb[both] = [255, 255, 0]

tp = (pred_np == 1) & (gt_np == 1)
fp = (pred_np == 1) & (gt_np == 0)
fn = (pred_np == 0) & (gt_np == 1)
error_map = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
error_map[tp] = [0, 255, 0]
error_map[fp] = [255, 0, 0]
error_map[fn] = [0, 0, 255]

fig, axes = plt.subplots(2, 4, figsize=(20, 10), dpi=300)
fig.suptitle(f"Publication Figure — {MODEL_NAME}\n{best_sample['img_name']} | Dice: {best_sample['dice']:.4f}, IoU: {best_sample['iou']:.4f}",
             fontsize=14, fontweight='bold')

panels = [
    (axes[0, 0], img_np, "Original Image", 'gray'),
    (axes[0, 1], gt_np, "Ground Truth", 'gray'),
    (axes[0, 2], pred_np, "Prediction", 'gray'),
    (axes[0, 3], boundary_rgb, "Boundary Overlay (GT/Pred)", None),
    (axes[1, 0], error_map, "Error Map (TP/FP/FN)", None),
    (axes[1, 1], cam_overlay, "Grad-CAM Overlay", None),
    (axes[1, 2], confidence, "Confidence Map", 'RdYlGn'),
    (axes[1, 3], entropy, "Entropy (Uncertainty)", 'hot'),
]

for ax, data, title, cmap in panels:
    if cmap is not None:
        ax.imshow(data, cmap=cmap)
    else:
        ax.imshow(data)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
save_path = os.path.join(RESULTS_ROOT, 'Publication', 'publication_figure.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")
plt.close(fig)

---
## 25 · Final Summary Report

In [ ]:
dice_vals = [s['dice'] for s in per_sample_results]
iou_vals = [s['iou'] for s in per_sample_results]
prec_vals = [s['precision'] for s in per_sample_results]
rec_vals = [s['recall'] for s in per_sample_results]
acc_vals = [s['accuracy'] for s in per_sample_results]
f1_vals = [s['f1'] for s in per_sample_results]
spec_vals = [s['specificity'] for s in per_sample_results]
sens_vals = [s['sensitivity'] for s in per_sample_results]
hd95_vals = [s['hd95'] for s in per_sample_results if not np.isnan(s['hd95'])]
assd_vals = [s['assd'] for s in per_sample_results if not np.isnan(s['assd'])]

def summary_stats(arr):
    return {
        'Mean': np.mean(arr),
        'Median': np.median(arr),
        'Std': np.std(arr),
        'Min': np.min(arr),
        'Max': np.max(arr),
    }

print("=" * 70)
print(f"  FINAL SUMMARY — {MODEL_NAME}")
print("=" * 70)
print(f"  {'Metric':<20} {'Mean':>10} {'Median':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print(f"  {'-'*60}")

all_metrics = {
    'Dice': dice_vals, 'IoU': iou_vals, 'Precision': prec_vals,
    'Recall': rec_vals, 'Accuracy': acc_vals, 'F1': f1_vals,
    'Specificity': spec_vals, 'Sensitivity': sens_vals,
    'HD95': hd95_vals, 'ASSD': assd_vals,
}

for metric_name, arr in all_metrics.items():
    stats = summary_stats(arr)
    print(f"  {metric_name:<20} {stats['Mean']:>10.4f} {stats['Median']:>10.4f} {stats['Std']:>10.4f} {stats['Min']:>10.4f} {stats['Max']:>10.4f}")

print("=" * 70)
print(f"  {'FPS':<20} {fps:>10.2f}")
print(f"  {'Avg Inference Time (ms)':<20} {avg_inference_ms:>10.2f}")
if 'gflops' in locals() and not np.isnan(gflops):
    print(f"  {'FLOPs (GFLOPs)':<20} {gflops:>10.4f}")
    print(f"  {'MACs (GMACs)':<20} {gmacs:>10.4f}")
print(f"  {'Model Size (MB)':<20} {model_size_mb:>10.2f}")
print(f"  {'Total Parameters':<20} {total_params:>10,}")
print(f"  {'Trainable Parameters':<20} {trainable_params:>10,}")
print("=" * 70)
print("\nAll results saved in:")
print(f"  {os.path.abspath(RESULTS_ROOT)}/")
